In [1]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

ytt_api = YouTubeTranscriptApi()

In [3]:
video_id = 'syE8f_ow0iM'
print(ytt_api.list(video_id=video_id))

For this video (syE8f_ow0iM) transcripts are available in the following languages:

(MANUALLY CREATED)
None

(GENERATED)
 - en ("English (auto-generated)")[TRANSLATABLE]
 - fr-FR ("French (France) (auto-generated)")[TRANSLATABLE]
 - de-DE ("German (Germany) (auto-generated)")[TRANSLATABLE]
 - hi ("Hindi (auto-generated)")[TRANSLATABLE]
 - id ("Indonesian (auto-generated)")[TRANSLATABLE]
 - it ("Italian (auto-generated)")[TRANSLATABLE]
 - pt-BR ("Portuguese (Brazil) (auto-generated)")[TRANSLATABLE]
 - es-US ("Spanish (United States) (auto-generated)")[TRANSLATABLE]

(TRANSLATION LANGUAGES)
 - ar ("Arabic")
 - zh-Hant ("Chinese (Traditional)")
 - nl ("Dutch")
 - en ("English")
 - fr ("French")
 - de ("German")
 - hi ("Hindi")
 - id ("Indonesian")
 - it ("Italian")
 - ja ("Japanese")
 - ko ("Korean")
 - pt ("Portuguese")
 - ru ("Russian")
 - es ("Spanish")
 - th ("Thai")
 - uk ("Ukrainian")
 - vi ("Vietnamese")


In [4]:
try:
    transcript_list = ytt_api.fetch(video_id=video_id)
    print('Transcript list length: ', len(transcript_list), end='\n\n')
    print(transcript_list[0])
except TranscriptsDisabled:
    print('Transcript disabled for this video')

Transcript list length:  2404

FetchedTranscriptSnippet(text='What is up ladies and gentlemen? Welcome', start=0.24, duration=4.32)


In [5]:
transcript = ' '.join(chunk.text for chunk in transcript_list)
print('Total words in the transcript: ', len(transcript))

Total words in the transcript:  86691


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

In [7]:
chunks = splitter.create_documents(texts=[transcript])
print('Number of chunks: ', len(chunks), end='\n\n')
print('First chunk: ', chunks[0])


Number of chunks:  134

First chunk:  page_content='What is up ladies and gentlemen? Welcome back to Fudge Muppet. I'm Scott here with Michael and today we're talking all the law you need to know for Elder Scrolls 6. >> Yes. And I have a feeling this is going to be a series. Yeah. Series that people have been wanting for a long time. Dare I say the return of the Elder Scrolls 6 podcast. Maybe potentially probably. Now, we've been waiting for Elder Scrolls 6 for a very, very, very long time. In fact, since high school. We made this channel when we were wearing school uniforms and we would joke about how we'll have kids playing The Elder Scrolls 6. Well, I can tell you what, we need to wear glasses for this episode. >> We're not going to be able to read >> cringe glasses reveal, but we literally need to wear glasses. We have gray hairs on the'


___
Creating vector store:

In [8]:
from langchain_openai import OpenAIEmbeddings
import os

# embedder = OpenAIEmbeddings(
#     model='local qwen3:0.6b-f16',
#     base_url='http://127.0.0.1:8081/v1',
#     api_key='not needed' # type: ignore
# )

embedder = OpenAIEmbeddings(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",     # type: ignore
    model="local-bge-m3",
    check_embedding_ctx_length=False
)

# embedder = OpenAIEmbeddings(
#     model='nvidia/nemotron-3-embed-1b:free', 
#     base_url="https://openrouter.ai/api/v1",
#     check_embedding_ctx_length=False,
#     api_key=os.environ['OPENROUTER_API_KEY'] # type: ignore
# )

In [9]:
from dotenv import load_dotenv
load_dotenv()


from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedder,
    collection_name='chroma_collection',
)



Indexing completed, Retrieval next

In [10]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k':10
    }
)

Augmentation

In [11]:
from langchain_core.prompts import PromptTemplate

llm_prompt_template = PromptTemplate(
    template="""
You are a question-answering assistant.

Answer the question using ONLY the information contained in the CONTEXT below.

IMPORTANT:
- Do not use your own knowledge.
- Do not assume facts that are not present in the context.
- If the context does not contain enough information to answer the question, say "I don't know based on the provided context."
- The phrase "Great War" may refer to something specific in the provided context. Use the meaning given by the context, not your general knowledge.
- Do not worry about the source being asked from. For example, if the user asks about a "video" or a "blog", do not worry about the video itself. Assume the context given is from the video, and answer the remaining question accordingly. You will not have access to the source itself, just the transcript's relevant context from that source.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""",
    input_variables=["context", "question"],
    validate_template=True
)

In [12]:
user_query = 'The Great War'

retrieved_docs = retriever.invoke(user_query)
context = ' '.join(chunk.page_content for chunk in retrieved_docs)

final_prompt = llm_prompt_template.invoke({'context': context, 'question':user_query})

Next, generation:

In [13]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

# llm = ChatOpenAI(
#     model='local qwen3.5:4b-8b',
#     base_url='http://127.0.0.1:8080/v1',
#     api_key='not needed' # type: ignore
# )

from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model='nvidia/nemotron-3.5-lightning:free'
)

Trying with a really good model using openrouter. Will later compare this output to a local qwen 3.5:4b model

In [14]:
result = llm.invoke(final_prompt)
print(result.content)

Based on the provided context, the Great War is described as a conflict that begins towards the end of the Fourth Era (171) between the Old Mary Dominion and the Empire. It was triggered when the Old Mary Dominion demanded the emperor stop worshipping Talos, cede Hammerfell, and pay a large sum of money. The emperor refused (summarized by the "Titus meme says the second says no"), leading the Dominion to invade. Their original goal was to take Hammerfell; they sent forces into the southwest of Siridel (around Anvil and Kabach) to draw imperial troops away, then aggressively attacked south Hammerfell, causing the Legion to be overwhelmed and forced to retreat across the Alik'r desert during an event called the "March of Thirst"—a long, slow war of attrition. Key figures mentioned include Lady Arinelia as the Dominion's commander/general and Legion general Desianis leading resistance. The war expanded when the emperor called forces back to the Battle of the Red Ring to retake the Imperia

In [15]:
retrieved_docs

[Document(id='15ad86f8-fcea-45c2-b623-be71d229eb7f', metadata={}, page_content="but anyway so we should all be pretty familiar with the great war so it starts off towards the end of the fourth era 171 and it's basically the old Mary Dominion show up to Siridil and they're like to the emperor and we're like we killed all of your blades agents within all of the uh all of Veilenwood and any old Mary Dominion territories and Uh, guess what? Give us the south of Hammerfell. Stop worshipping Talos. Give us a ton of money. Just do it. Like straight up this Titus meme says the second says no. And then so the Great War begins. And it really starts off with, and we can read the the Great War book talking about it. It starts off as if the original goal of the Old Mary Dominion was kind of just to take Hammerfell and they sent a force into the uh southwest of Siridel around"),
 Document(id='4254bdda-ffa8-47d1-8dab-03fd48f140a0', metadata={}, page_content="on there's all these different moving piec

In [16]:
# 
local_llm = ChatOpenAI(
    model='local qwen3.5:4b-8b',
    base_url='http://127.0.0.1:8080/v1',
    api_key='not needed' # type: ignore
)

### Clearly, keeping everything else the exact same and only trying slightly different models, it is clear that the cloud model is much better

This was tried with reasoning off on the local qwen3.5, but with reasoning on, it was an even bigger problem. The model just kept stuck in a loop of "Wait, have I checked this", and "Yes, but then have I done this again". So decided to try with reasoning off

___
Building a chain for this. Will use openrouter models for this, although anyone with a better GPU can use a local LLM and get good performance. My preference will always be local models, but since I cannot run good models on my RTX3050 with 4GB VRAM, I will use Openrouter here  

Promp -> LLM -> Output: Very simple chain, just a RunnableSequence <br><br>

Will need another parallel chain for the prompt though, since you have two input fields for the prompt, the user question and the retrieved chunks

In [19]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

from typing import List
from langchain_core.documents import Document

In [20]:
def join_retrieved_docs(retrieved_docs: List[Document]) -> str:
    return ' '.join(chunk.page_content for chunk in retrieved_docs)

In [21]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(join_retrieved_docs),
    'question': RunnablePassthrough()
})

In [22]:
str_parser = StrOutputParser()

In [23]:
final_chain = parallel_chain | llm_prompt_template | llm | str_parser # type: ignore

In [24]:
user_query = input('Ask something')

result = final_chain.invoke(user_query)

In [25]:
print('Returned answer:\n', result)

Returned answer:
 Yes, based on the provided context, the Redguards are a main topic of discussion. The transcript is dedicated to an overview of Redguard lore, covering their origin on the continent of Yakuda, their migration to Hammerfell, the split between Crown and Forebear factions, their cultural practices (ancestor worship, sailing, warfare, armor and attire suited to desert environments), their attitudes toward necromancy and the undead, their interaction with the Empire, and their planned deeper exploration in future Elder Scrolls 6 content. The discussion consistently returns to Redguard history, society, and their role within the broader Elder Scrolls setting.
